# Silver Layer - Data Transformation

This notebook reads data from the bronze layer (`workspace.bronze`), applies transformations, checks for data quality issues (nulls, duplicates), and creates clean, business-ready tables in the silver layer (`workspace.silver`).

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Create silver schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

print("✅ Environment setup complete")
print("📦 Silver schema: workspace.silver")

In [0]:
print("Loading tables from bronze layer...\n")

# Load fact table
fact_sales_bronze = spark.table("workspace.bronze.fact_sales")

# Load dimension tables
dim_customer_bronze = spark.table("workspace.bronze.dim_customer")
dim_date_bronze = spark.table("workspace.bronze.dim_date")
dim_region_bronze = spark.table("workspace.bronze.dim_region")
dim_product_bronze = spark.table("workspace.bronze.dim_product")
dim_product_subcat_bronze = spark.table("workspace.bronze.dim_product_subcategory")
dim_product_cat_bronze = spark.table("workspace.bronze.dim_product_category")

print("✅ All bronze tables loaded successfully\n")
print("Bronze layer record counts:")
print(f"  • fact_sales: {fact_sales_bronze.count():,} rows")
print(f"  • dim_customer: {dim_customer_bronze.count():,} rows")
print(f"  • dim_date: {dim_date_bronze.count():,} rows")
print(f"  • dim_region: {dim_region_bronze.count():,} rows")
print(f"  • dim_product: {dim_product_bronze.count():,} rows")
print(f"  • dim_product_subcategory: {dim_product_subcat_bronze.count():,} rows")
print(f"  • dim_product_category: {dim_product_cat_bronze.count():,} rows")

In [0]:
print("\n" + "="*80)
print("DATA QUALITY CHECKS - NULL VALUES")
print("="*80)

def check_nulls(df, table_name):
    """Check for null values in each column"""
    print(f"\n📊 {table_name}:")
    null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).collect()[0].asDict()
    
    has_nulls = False
    for column, null_count in null_counts.items():
        if null_count > 0:
            has_nulls = True
            print(f"  ⚠️  {column}: {null_count:,} null values ({null_count/df.count()*100:.2f}%)")
    
    if not has_nulls:
        print("  ✅ No null values found")
    
    return null_counts

# Check nulls in all tables
null_report_fact = check_nulls(fact_sales_bronze, "fact_sales")
null_report_customer = check_nulls(dim_customer_bronze, "dim_customer")
null_report_date = check_nulls(dim_date_bronze, "dim_date")
null_report_region = check_nulls(dim_region_bronze, "dim_region")
null_report_product = check_nulls(dim_product_bronze, "dim_product")
null_report_subcat = check_nulls(dim_product_subcat_bronze, "dim_product_subcategory")
null_report_cat = check_nulls(dim_product_cat_bronze, "dim_product_category")

print("\n✅ Null check completed")

In [0]:
print("\n" + "="*80)
print("DATA QUALITY CHECKS - DUPLICATE RECORDS")
print("="*80)

def check_duplicates(df, key_columns, table_name):
    """Check for duplicate records based on key columns"""
    total_rows = df.count()
    distinct_rows = df.select(key_columns).distinct().count()
    duplicates = total_rows - distinct_rows
    
    print(f"\n📊 {table_name}:")
    print(f"  • Total rows: {total_rows:,}")
    print(f"  • Distinct rows (by {', '.join(key_columns)}): {distinct_rows:,}")
    
    if duplicates > 0:
        print(f"  ⚠️  Duplicates found: {duplicates:,} duplicate records")
    else:
        print(f"  ✅ No duplicates found")
    
    return duplicates

# Check duplicates in all tables
dup_fact = check_duplicates(fact_sales_bronze, ["OrderNumber"], "fact_sales")
dup_customer = check_duplicates(dim_customer_bronze, ["CustomerKey"], "dim_customer")
dup_date = check_duplicates(dim_date_bronze, ["DateKey"], "dim_date")
dup_region = check_duplicates(dim_region_bronze, ["GeographyKey"], "dim_region")
dup_product = check_duplicates(dim_product_bronze, ["ProductKey"], "dim_product")
dup_subcat = check_duplicates(dim_product_subcat_bronze, ["ProductSubcategoryKey"], "dim_product_subcategory")
dup_cat = check_duplicates(dim_product_cat_bronze, ["ProductCategoryKey"], "dim_product_category")

print("\n✅ Duplicate check completed")

In [0]:
print("\n" + "="*80)
print("TRANSFORMING DIMENSION TABLES")
print("="*80)

# 1. Transform dim_customer
print("\n1. Transforming dim_customer...")
dim_customer_silver = (
    dim_customer_bronze
    .dropDuplicates(["CustomerKey"])  # Remove duplicates
    .filter(col("CustomerKey").isNotNull())  # Remove null keys
    .withColumn("CustomerName", coalesce(col("CustomerName"), lit("Unknown")))  # Handle null names
    .withColumn("MaritalStatus", coalesce(col("MaritalStatus"), lit("U")))  # Unknown
    .withColumn("Gender", coalesce(col("Gender"), lit("U")))  # Unknown
    .withColumn("GeographyKey", coalesce(col("GeographyKey"), lit(-1)))  # Default -1 for unknown
)
print(f"   ✓ Cleaned: {dim_customer_silver.count():,} rows")

# 2. Transform dim_date
print("\n2. Transforming dim_date...")
dim_date_silver = (
    dim_date_bronze
    .dropDuplicates(["DateKey"])
    .filter(col("DateKey").isNotNull())
    .withColumn("EnglishMonthName", coalesce(col("EnglishMonthName"), lit("Unknown")))
    .withColumn("CalendarYear", coalesce(col("CalendarYear"), lit(0)))
    # Add derived columns
    .withColumn("Quarter", quarter(col("FullDateAlternateKey")))
    .withColumn("Month", month(col("FullDateAlternateKey")))
    .withColumn("DayOfWeek", dayofweek(col("FullDateAlternateKey")))
    .withColumn("WeekOfYear", weekofyear(col("FullDateAlternateKey")))
)
print(f"   ✓ Cleaned and enriched: {dim_date_silver.count():,} rows")

# 3. Transform dim_region
print("\n3. Transforming dim_region...")
dim_region_silver = (
    dim_region_bronze
    .dropDuplicates(["GeographyKey"])
    .filter(col("GeographyKey").isNotNull())
    .withColumn("City", coalesce(col("City"), lit("Unknown")))
    .withColumn("Region", coalesce(col("Region"), lit("Unknown")))
    .withColumn("Country", coalesce(col("Country"), lit("Unknown")))
)
print(f"   ✓ Cleaned: {dim_region_silver.count():,} rows")

# 4. Transform dim_product_category
print("\n4. Transforming dim_product_category...")
dim_product_cat_silver = (
    dim_product_cat_bronze
    .dropDuplicates(["ProductCategoryKey"])
    .filter(col("ProductCategoryKey").isNotNull())
    .withColumn("ProductCategoryName", coalesce(col("ProductCategoryName"), lit("Unknown Category")))
)
print(f"   ✓ Cleaned: {dim_product_cat_silver.count():,} rows")

# 5. Transform dim_product_subcategory
print("\n5. Transforming dim_product_subcategory...")
dim_product_subcat_silver = (
    dim_product_subcat_bronze
    .dropDuplicates(["ProductSubcategoryKey"])
    .filter(col("ProductSubcategoryKey").isNotNull())
    .withColumn("ProductSubcategoryName", coalesce(col("ProductSubcategoryName"), lit("Unknown Subcategory")))
    .withColumn("ProductCategoryKey", coalesce(col("ProductCategoryKey"), lit(-1)))
)
print(f"   ✓ Cleaned: {dim_product_subcat_silver.count():,} rows")

# 6. Transform dim_product
print("\n6. Transforming dim_product...")
dim_product_silver = (
    dim_product_bronze
    .dropDuplicates(["ProductKey"])
    .filter(col("ProductKey").isNotNull())
    .withColumn("Product_Name", coalesce(col("Product_Name"), lit("Unknown Product")))
    .withColumn("Color", coalesce(col("Color"), lit("N/A")))
    .withColumn("StandardCost", coalesce(col("StandardCost"), lit(0.0)))
    .withColumn("ListPrice", coalesce(col("ListPrice"), lit(0.0)))
    .withColumn("ProductSubcategoryKey", coalesce(col("ProductSubcategoryKey"), lit(-1)))
    # Business logic: Calculate profit margin
    .withColumn("ProfitMargin", 
        when(col("ListPrice") > 0, 
             ((col("ListPrice") - col("StandardCost")) / col("ListPrice") * 100))
        .otherwise(0.0))
)
print(f"   ✓ Cleaned and enriched: {dim_product_silver.count():,} rows")

print("\n✅ All dimension tables transformed")

In [0]:
print("\n" + "="*80)
print("TRANSFORMING FACT TABLE")
print("="*80)

print("\nTransforming fact_sales...")

# Clean and transform fact table
fact_sales_silver = (
    fact_sales_bronze
    # Remove duplicates based on OrderNumber
    .dropDuplicates(["OrderNumber"])
    # Remove records with null keys (can't join)
    .filter(col("ProductKey").isNotNull() & 
            col("OrderDateKey").isNotNull() & 
            col("CustomerKey").isNotNull())
    # Handle null quantities and prices
    .withColumn("OrderQuantity", coalesce(col("OrderQuantity"), lit(0)))
    .withColumn("List_Price", coalesce(col("List_Price"), lit(0.0)))
    .withColumn("Product_Cost", coalesce(col("Product_Cost"), lit(0.0)))
    .withColumn("Gender", coalesce(col("Gender"), lit("U")))
    # Business logic: Calculate revenue
    .withColumn("Revenue", col("OrderQuantity") * col("List_Price"))
    # Business logic: Calculate total cost
    .withColumn("TotalCost", col("OrderQuantity") * col("Product_Cost"))
    # Business logic: Calculate profit
    .withColumn("Profit", col("Revenue") - col("TotalCost"))
    # Business logic: Calculate profit margin percentage
    .withColumn("ProfitMarginPct", 
        when(col("Revenue") > 0, (col("Profit") / col("Revenue") * 100))
        .otherwise(0.0))
    # Add data quality flag
    .withColumn("IsValidRecord", 
        when((col("OrderQuantity") > 0) & (col("List_Price") >= 0), True)
        .otherwise(False))
)

print(f"   ✓ Cleaned and enriched: {fact_sales_silver.count():,} rows")
print(f"   ✓ Valid records: {fact_sales_silver.filter(col('IsValidRecord') == True).count():,}")
print(f"   ⚠️  Invalid records: {fact_sales_silver.filter(col('IsValidRecord') == False).count():,}")

print("\n✅ Fact table transformed")

In [0]:
print("\n" + "="*80)
print("CREATING ENRICHED FACT TABLE")
print("="*80)

print("\nJoining fact table with dimensions for enriched view...")

# Create enriched fact table with key dimension attributes
fact_sales_enriched = (
    fact_sales_silver
    # Join with customer
    .join(dim_customer_silver.select(
        col("CustomerKey"),
        col("CustomerName"),
        col("MaritalStatus").alias("CustomerMaritalStatus"),
        col("GeographyKey")
    ), "CustomerKey", "left")
    # Join with region
    .join(dim_region_silver.select(
        col("GeographyKey"),
        col("City"),
        col("Region"),
        col("Country")
    ), "GeographyKey", "left")
    # Join with date
    .join(dim_date_silver.select(
        col("DateKey").alias("OrderDateKey"),
        col("FullDateAlternateKey").alias("OrderDate"),
        col("EnglishMonthName").alias("OrderMonth"),
        col("CalendarYear").alias("OrderYear"),
        col("Quarter").alias("OrderQuarter")
    ), "OrderDateKey", "left")
    # Join with product
    .join(dim_product_silver.select(
        col("ProductKey"),
        col("Product_Name"),
        col("Color").alias("ProductColor"),
        col("ProductSubcategoryKey"),
        col("ProfitMargin").alias("ProductProfitMargin")
    ), "ProductKey", "left")
    # Join with subcategory
    .join(dim_product_subcat_silver.select(
        col("ProductSubcategoryKey"),
        col("ProductSubcategoryName"),
        col("ProductCategoryKey")
    ), "ProductSubcategoryKey", "left")
    # Join with category
    .join(dim_product_cat_silver.select(
        col("ProductCategoryKey"),
        col("ProductCategoryName")
    ), "ProductCategoryKey", "left")
    # Select and order columns
    .select(
        # Keys
        col("OrderNumber"),
        col("ProductKey"),
        col("CustomerKey"),
        col("OrderDateKey"),
        col("GeographyKey"),
        # Order details
        col("OrderDate"),
        col("OrderYear"),
        col("OrderQuarter"),
        col("OrderMonth"),
        col("OrderQuantity"),
        # Product details
        col("Product_Name"),
        col("ProductColor"),
        col("ProductSubcategoryName"),
        col("ProductCategoryName"),
        # Customer details
        col("CustomerName"),
        col("Gender"),
        col("CustomerMaritalStatus"),
        # Geography
        col("City"),
        col("Region"),
        col("Country"),
        # Financial metrics
        col("List_Price"),
        col("Product_Cost"),
        col("Revenue"),
        col("TotalCost"),
        col("Profit"),
        col("ProfitMarginPct"),
        col("ProductProfitMargin"),
        # Quality flag
        col("IsValidRecord")
    )
)

print(f"   ✓ Enriched fact table created: {fact_sales_enriched.count():,} rows")
print("\n✅ Enriched fact table ready")

In [0]:
print("\n" + "="*80)
print("SAVING SILVER LAYER TABLES")
print("="*80)

print("\nWriting tables to workspace.silver...")

# Save dimension tables
print("\n1. Saving dim_customer...")
dim_customer_silver.write.mode("overwrite").saveAsTable("workspace.silver.dim_customer")
print("   ✓ workspace.silver.dim_customer created")

print("\n2. Saving dim_date...")
dim_date_silver.write.mode("overwrite").saveAsTable("workspace.silver.dim_date")
print("   ✓ workspace.silver.dim_date created")

print("\n3. Saving dim_region...")
dim_region_silver.write.mode("overwrite").saveAsTable("workspace.silver.dim_region")
print("   ✓ workspace.silver.dim_region created")

print("\n4. Saving dim_product_category...")
dim_product_cat_silver.write.mode("overwrite").saveAsTable("workspace.silver.dim_product_category")
print("   ✓ workspace.silver.dim_product_category created")

print("\n5. Saving dim_product_subcategory...")
dim_product_subcat_silver.write.mode("overwrite").saveAsTable("workspace.silver.dim_product_subcategory")
print("   ✓ workspace.silver.dim_product_subcategory created")

print("\n6. Saving dim_product...")
dim_product_silver.write.mode("overwrite").saveAsTable("workspace.silver.dim_product")
print("   ✓ workspace.silver.dim_product created")

print("\n7. Saving fact_sales...")
fact_sales_silver.write.mode("overwrite").saveAsTable("workspace.silver.fact_sales")
print("   ✓ workspace.silver.fact_sales created")

print("\n8. Saving fact_sales_enriched...")
fact_sales_enriched.write.mode("overwrite").saveAsTable("workspace.silver.fact_sales_enriched")
print("   ✓ workspace.silver.fact_sales_enriched created")

print("\n" + "="*80)
print("✅ ALL SILVER LAYER TABLES SAVED SUCCESSFULLY")
print("="*80)

In [0]:
print("\n" + "="*80)
print("SILVER LAYER SUMMARY")
print("="*80)

# Display all silver tables
print("\n📋 Tables in workspace.silver:")
tables = spark.sql("SHOW TABLES IN workspace.silver")
display(tables)

# Display row counts
print("\n📊 DIMENSION TABLES:")
print(f"  • dim_customer: {spark.table('workspace.silver.dim_customer').count():,} rows")
print(f"  • dim_date: {spark.table('workspace.silver.dim_date').count():,} rows")
print(f"  • dim_region: {spark.table('workspace.silver.dim_region').count():,} rows")
print(f"  • dim_product: {spark.table('workspace.silver.dim_product').count():,} rows")
print(f"  • dim_product_subcategory: {spark.table('workspace.silver.dim_product_subcategory').count():,} rows")
print(f"  • dim_product_category: {spark.table('workspace.silver.dim_product_category').count():,} rows")

print("\n📊 FACT TABLES:")
print(f"  • fact_sales: {spark.table('workspace.silver.fact_sales').count():,} rows")
print(f"  • fact_sales_enriched: {spark.table('workspace.silver.fact_sales_enriched').count():,} rows")

# Display sample data quality metrics
print("\n📈 BUSINESS METRICS SUMMARY:")
metrics = spark.sql("""
    SELECT 
        COUNT(*) as total_orders,
        ROUND(SUM(Revenue), 2) as total_revenue,
        ROUND(SUM(Profit), 2) as total_profit,
        ROUND(AVG(ProfitMarginPct), 2) as avg_profit_margin_pct,
        COUNT(DISTINCT CustomerKey) as unique_customers,
        COUNT(DISTINCT ProductKey) as unique_products
    FROM workspace.silver.fact_sales
""")
display(metrics)

print("\n✅ Silver layer transformation completed successfully!")
print("\n💡 Next steps:")
print("  1. Create gold layer for aggregated analytics")
print("  2. Build dashboards using silver.fact_sales_enriched")
print("  3. Set up data quality monitoring")

In [0]:
print("\n" + "="*80)
print("CREATING COMPREHENSIVE FINAL JOINED TABLE")
print("="*80)

print("\nJoining all dimension and fact tables...")

# Create a comprehensive final table with ALL columns from all tables
final_joined_table = (
    fact_sales_silver
    # Join with dim_customer - get all customer attributes
    .join(
        dim_customer_silver,
        fact_sales_silver.CustomerKey == dim_customer_silver.CustomerKey,
        "inner"
    )
    # Join with dim_region - get all geography attributes
    .join(
        dim_region_silver,
        dim_customer_silver.GeographyKey == dim_region_silver.GeographyKey,
        "left"
    )
    # Join with dim_date - get all date attributes
    .join(
        dim_date_silver,
        fact_sales_silver.OrderDateKey == dim_date_silver.DateKey,
        "inner"
    )
    # Join with dim_product - get all product attributes
    .join(
        dim_product_silver,
        fact_sales_silver.ProductKey == dim_product_silver.ProductKey,
        "inner"
    )
    # Join with dim_product_subcategory - get subcategory attributes
    .join(
        dim_product_subcat_silver,
        dim_product_silver.ProductSubcategoryKey == dim_product_subcat_silver.ProductSubcategoryKey,
        "left"
    )
    # Join with dim_product_category - get category attributes
    .join(
        dim_product_cat_silver,
        dim_product_subcat_silver.ProductCategoryKey == dim_product_cat_silver.ProductCategoryKey,
        "left"
    )
    # Select all relevant columns in a logical order
    .select(
        # FACT TABLE - Order Information
        fact_sales_silver.OrderNumber.alias("OrderNumber"),
        fact_sales_silver.OrderQuantity.alias("OrderQuantity"),
        fact_sales_silver.List_Price.alias("UnitPrice"),
        fact_sales_silver.Product_Cost.alias("UnitCost"),
        
        # FACT TABLE - Calculated Metrics
        fact_sales_silver.Revenue.alias("Revenue"),
        fact_sales_silver.TotalCost.alias("TotalCost"),
        fact_sales_silver.Profit.alias("Profit"),
        fact_sales_silver.ProfitMarginPct.alias("ProfitMarginPct"),
        fact_sales_silver.IsValidRecord.alias("IsValidRecord"),
        
        # DATE DIMENSION - All date attributes
        dim_date_silver.DateKey.alias("DateKey"),
        dim_date_silver.FullDateAlternateKey.alias("OrderDate"),
        dim_date_silver.CalendarYear.alias("Year"),
        dim_date_silver.Quarter.alias("Quarter"),
        dim_date_silver.Month.alias("Month"),
        dim_date_silver.EnglishMonthName.alias("MonthName"),
        dim_date_silver.WeekOfYear.alias("WeekOfYear"),
        dim_date_silver.DayOfWeek.alias("DayOfWeek"),
        
        # CUSTOMER DIMENSION - All customer attributes
        dim_customer_silver.CustomerKey.alias("CustomerKey"),
        dim_customer_silver.CustomerName.alias("CustomerName"),
        dim_customer_silver.Gender.alias("CustomerGender"),
        dim_customer_silver.MaritalStatus.alias("CustomerMaritalStatus"),
        
        # GEOGRAPHY DIMENSION - All geography attributes
        dim_region_silver.GeographyKey.alias("GeographyKey"),
        dim_region_silver.City.alias("City"),
        dim_region_silver.Region.alias("Region"),
        dim_region_silver.Country.alias("Country"),
        
        # PRODUCT DIMENSION - All product attributes
        dim_product_silver.ProductKey.alias("ProductKey"),
        dim_product_silver.Product_Name.alias("ProductName"),
        dim_product_silver.Color.alias("ProductColor"),
        dim_product_silver.StandardCost.alias("ProductStandardCost"),
        dim_product_silver.ListPrice.alias("ProductListPrice"),
        dim_product_silver.ProfitMargin.alias("ProductProfitMargin"),
        
        # PRODUCT SUBCATEGORY DIMENSION
        dim_product_subcat_silver.ProductSubcategoryKey.alias("ProductSubcategoryKey"),
        dim_product_subcat_silver.ProductSubcategoryName.alias("ProductSubcategory"),
        
        # PRODUCT CATEGORY DIMENSION
        dim_product_cat_silver.ProductCategoryKey.alias("ProductCategoryKey"),
        dim_product_cat_silver.ProductCategoryName.alias("ProductCategory")
    )
)

print(f"\n✓ Final joined table created with {final_joined_table.count():,} rows")
print(f"✓ Total columns: {len(final_joined_table.columns)}")

print("\n📋 Column structure:")
print("  • Order metrics: 9 columns")
print("  • Date attributes: 8 columns")
print("  • Customer attributes: 4 columns")
print("  • Geography attributes: 4 columns")
print("  • Product attributes: 6 columns")
print("  • Category hierarchy: 4 columns")
print(f"  • TOTAL: {len(final_joined_table.columns)} columns")

print("\n✅ Comprehensive joined table ready!")

In [0]:
print("\n" + "="*80)
print("FINAL JOINED TABLE - SAMPLE DATA")
print("="*80)

# Display schema
print("\n📑 Table Schema:")
final_joined_table.printSchema()

# Display sample rows
print("\n📊 Sample Records (first 10 rows):")
display(final_joined_table.limit(10))

In [0]:
print("\n" + "="*80)
print("SAVING FINAL JOINED TABLE")
print("="*80)

print("\nSaving comprehensive joined table to workspace.silver.final_sales_complete...")

# Save the final comprehensive table
final_joined_table.write.mode("overwrite").saveAsTable("workspace.silver.final_sales_complete")

print("✓ workspace.silver.final_sales_complete created")
print(f"\n✅ Final table saved with {final_joined_table.count():,} rows and {len(final_joined_table.columns)} columns")

# Verify the saved table
print("\n📈 Verification:")
verify_query = spark.sql("""
    SELECT 
        COUNT(*) as total_records,
        COUNT(DISTINCT OrderNumber) as unique_orders,
        COUNT(DISTINCT CustomerKey) as unique_customers,
        COUNT(DISTINCT ProductKey) as unique_products,
        COUNT(DISTINCT Country) as unique_countries,
        ROUND(SUM(Revenue), 2) as total_revenue,
        ROUND(SUM(Profit), 2) as total_profit
    FROM workspace.silver.final_sales_complete
""")
display(verify_query)

print("\n" + "="*80)
print("✅ FINAL JOINED TABLE SUCCESSFULLY CREATED AND SAVED!")
print("="*80)
print("\n💡 Table: workspace.silver.final_sales_complete")
print("💡 This table contains all joined dimensions and fact data in one place")
print("💡 Ready for analysis, reporting, and dashboard creation!")